# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
A line chart fits because the question is about change over time. Every continent is set to 100 in 1970 so they compare fairly, and Africa is orange while the rest are grey so the fastest grower stands out.

In [3]:
# Q1
# YOUR CODE HERE
cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
cont['Index'] = cont.groupby('Continent')['Population'].transform(lambda s: s / s.iloc[0] * 100)

colors = {c: '#BBBBBB' for c in cont['Continent'].unique()}
colors['Africa'] = '#D55E00'

fig = px.line(cont, x='Year', y='Index', color='Continent',
              color_discrete_map=colors, template='plotly_white',
              title='Africa has grown fastest: its population is now about 3.5x its 1970 size')
fig.update_layout(showlegend=False, yaxis_title='Population (1970 = 100)', xaxis_title=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

last = cont[cont['Year'] == 2022]
for _, r in last.iterrows():
    fig.add_annotation(x=2022, y=r['Index'], text=r['Continent'], showarrow=False, xanchor='left')
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
A sorted horizontal bar chart works best for a top 15 ranking. African countries are orange and the others grey, so you can count them quickly.

In [4]:
# Q2
# YOUR CODE HERE
df['Added'] = df['2022 Population'] - df['1970 Population']
top = df.nlargest(15, 'Added').sort_values('Added')
top['Region'] = np.where(top['Continent'] == 'Africa', 'Africa', 'Other')
n_africa = (top['Region'] == 'Africa').sum()

fig = px.bar(top, x='Added', y='Country', color='Region', orientation='h',
             color_discrete_map={'Africa': '#D55E00', 'Other': '#BBBBBB'},
             template='plotly_white',
             title=f'{n_africa} of the 15 countries that added the most people since 1970 are in Africa')
fig.update_layout(showlegend=False, xaxis_title='People added, 1970–2022', yaxis_title=None)
fig.update_xaxes(showgrid=False)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
A scatter plot fits because the question is about the relationship between area and population. Log scales and continent colours show the pattern, and only outliers like China, India and Russia are labelled.

In [5]:
# Q3
# YOUR CODE HERE
okabe = ['#E69F00', '#56B4E9', '#009E73', '#0072B2', '#D55E00', '#CC79A7']
outliers = ['China', 'India', 'Russia', 'Canada', 'Greenland', 'Bangladesh', 'Singapore', 'Australia']
df['Label'] = np.where(df['Country'].isin(outliers), df['Country'], '')

fig = px.scatter(df, x='Area', y='2022 Population', color='Continent',
                 text='Label', hover_name='Country', log_x=True, log_y=True,
                 color_discrete_sequence=okabe, template='plotly_white',
                 title='Larger countries tend to have more people, but China, India and Russia break the pattern')
fig.update_traces(textposition='top center', marker=dict(size=8, opacity=0.8))
fig.update_layout(xaxis_title='Land area (km², log)', yaxis_title='Population 2022 (log)')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
A box plot shows the spread of density, not just the average. It gives the middle, the range and the outliers for each continent, and a log scale handles the huge differences in values.

In [6]:
# Q4
# YOUR CODE HERE
fig = px.box(df, x='Continent', y='Density', points='all', hover_name='Country',
             log_y=True, template='plotly_white',
             title='Density varies hugely within every continent, most of all in Asia')
fig.update_traces(marker_color='#0072B2', line_color='#555555')
fig.update_layout(xaxis_title=None, yaxis_title='People per km² (log scale)')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
A choropleth map shows where countries grew or shrank. Colour shows the percentage change from 2000 to 2022, on a blue-red scale centred on zero that avoids red/green.

In [7]:
# Q5
# YOUR CODE HERE
df['Change'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100

fig = px.choropleth(df, locations='CCA3', color='Change', hover_name='Country',
                    color_continuous_scale='RdBu', color_continuous_midpoint=0,
                    range_color=[-30, 150], template='plotly_white',
                    title='Most countries grew since 2000; a few in Eastern Europe shrank')
fig.update_layout(coloraxis_colorbar_title='% change')
fig.update_geos(showframe=False, showcoastlines=False)
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
A treemap shows continents and the countries inside them in one figure. Bigger rectangles mean more people, so Asia, China and India stand out.

In [8]:
# Q6
# YOUR CODE HERE
okabe = ['#E69F00', '#56B4E9', '#009E73', '#0072B2', '#D55E00', '#CC79A7']

fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'],
                 values='2022 Population', color='Continent',
                 color_discrete_sequence=okabe, template='plotly_white',
                 title='Asia holds most of the world, and China and India dominate it')
fig.update_traces(textinfo='label+percent root')
fig.update_layout(margin=dict(t=60, l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
A 100% stacked bar chart fits because the question is about shares of the world total over time. The other continents are grouped as “Rest of the world” in grey so Africa, Asia and Europe stand

In [9]:
# Q7
# YOUR CODE HERE
d = df_long.copy()
d['Group'] = np.where(d['Continent'].isin(['Africa', 'Asia', 'Europe']), d['Continent'], 'Rest of the world')

g = d.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
g['Share'] = g['Population'] / g.groupby('Year')['Population'].transform('sum') * 100
g['Year'] = g['Year'].astype(str)

fig = px.bar(g, x='Year', y='Share', color='Group', template='plotly_white',
             color_discrete_map={'Africa': '#D55E00', 'Asia': '#0072B2',
                                 'Europe': '#56B4E9', 'Rest of the world': '#BBBBBB'},
             title="Africa's share of the world is rising while Europe's is shrinking")
fig.update_layout(xaxis_title=None, yaxis_title='Share of world population (%)', legend_title=None)
fig.update_yaxes(showgrid=False)
fig.show()